# Construction de la table mots / synsets

À partir de la liste des mots les plus fréquents du français (`../data/data_toyes.txt`, colonnes : fréquence, mot), on construit :

1. une **table** associant à chaque mot français ses synsets WordNet ;
2. la liste des **concepts** (futurs nœuds du graphe), dédoublonnés par définition.

In [1]:
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /home/JP/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/JP/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 1. Lecture des mots

In [2]:
df = pd.read_csv('../data/data_toyes.txt', sep='\t', header=None, names=['freq', 'mot'])
mots = df['mot']
df.head()

,freq,mot
0,47221,tout
1,25388,grand
2,19008,petit
3,16081,même
4,14688,autre


## 2. Table `mot_fr / synset / mot_en / nature / num`

Une ligne par couple (mot, synset). `mot_en` est le premier lemme anglais du synset, `nature` la catégorie grammaticale et `num` le numéro du sens.

In [3]:
lignes = []
for mot in mots:
    for syn in wn.synsets(mot, lang='fra'):
        lignes.append({
            'mot_fr': mot,
            'synset': syn.name(),
            'mot_en': syn.lemma_names()[0],
            'nature': syn.pos(),
            'num': syn.name().split('.')[-1],
        })

df_expanded = pd.DataFrame(lignes, columns=['mot_fr', 'synset', 'mot_en', 'nature', 'num'])
df_expanded

,mot_fr,synset,mot_en,nature,num
0,tout,whole.n.02,whole,n,02
1,tout,wholly.r.01,wholly,r,01
2,tout,anywhere.r.01,anywhere,r,01
3,tout,so.r.07,so,r,07
4,tout,all.s.02,all,s,02
...,...,...,...,...,...
9041,composer,consist.v.02,consist,v,02
9042,enlever,remove.v.01,remove,v,01
9043,enlever,take_off.v.02,take_off,v,02
9044,enlever,remove.v.08,remove,v,08


## 3. Concepts (nœuds)

Plusieurs synsets peuvent avoir la même définition ; on ne garde que le premier de chaque définition.

In [4]:
concepts = {}
for mot in mots:
    for syn in wn.synsets(mot, lang='fra'):
        concepts.setdefault(syn.definition(), {
            'Major': syn.name(),
            'Definition': syn.definition(),
            'Synonymes': syn.lemma_names(),
        })
list_concept = list(concepts.values())
print(len(list_concept), 'concepts')
list_concept[:3]

6215 concepts


[{'Major': 'whole.n.02',
  'Definition': 'an assemblage of parts that is regarded as a single entity',
  'Synonymes': ['whole', 'unit']},
 {'Major': 'wholly.r.01',
  'Definition': "to a complete degree or to the full or entire extent (`whole' is often used informally for `wholly')",
  'Synonymes': ['wholly',
   'entirely',
   'completely',
   'totally',
   'all',
   'altogether',
   'whole']},
 {'Major': 'anywhere.r.01',
  'Definition': "at or in or to any place; ; (`anyplace' is used informally for `anywhere')",
  'Synonymes': ['anywhere', 'anyplace']}]